# Week 10 실습 · pandas로 데이터 불러오기와 탐색

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week10/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. `Series`와 `DataFrame`의 구조(인덱스·열·자료형)를 이해합니다.
2. `read_csv`로 데이터를 읽고 `head`·`info`·`describe`·`value_counts`로 **구조와 품질**을 탐색합니다.
3. 열 선택, `loc`/`iloc`, **조건 필터**(`&`, `|`, `isin`, `between`), 정렬, 새 열 만들기를 익힙니다.
4. 8주차 중간 실기의 성적 분석을 **pandas로 다시** 해 보고 결과를 대조합니다.
5. AI 코드에서 자주 나오는 **`and` 사용 오류**와 **체인 인덱싱으로 값이 안 바뀌는 문제**를 확인합니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | Level 1 Series와 DataFrame | 10분 |
| 2 | Level 1 데이터 읽기와 첫 탐색 | 15분 |
| 3 | Level 1 선택·필터·정렬·새 열 | 25분 |
| 4 | Level 2 바꾸기 | 10분 |
| 5 | Level 3 성적 데이터 탐색 보고 | 30분 |
| 6 | Level 4 AI 코드의 pandas 함정 | 20분 |
| 7 | 정리 | 10분 |

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

print("pandas", pd.__version__)

---
## 1. Level 1 따라하기: Series와 DataFrame

| 구조 | 비유 | 구성 |
|---|---|---|
| `Series` | 이름표가 붙은 1차원 배열 | 값 + **인덱스** |
| `DataFrame` | 엑셀 시트 | 여러 `Series`(열)가 같은 인덱스를 공유 |

In [ ]:
s = pd.Series([78, 92, 65], index=["김민준", "이서연", "박지호"], name="midterm")
print(s)
print("이서연:", s["이서연"], "/ 평균:", s.mean())
print(s[s >= 70])                      # NumPy처럼 마스크 사용

In [ ]:
df_small = pd.DataFrame({
    "name": ["김민준", "이서연", "박지호"],
    "dept": ["컴퓨터공학과", "정보보안학과", "컴퓨터공학과"],
    "midterm": [78, 92, 65],
    "final": [85, 88, 70],
})
df_small

In [ ]:
print(df_small.shape, list(df_small.columns))
print(df_small.dtypes)
print(type(df_small["midterm"]))        # 열 하나 = Series

---
## 2. Level 1 따라하기: 데이터 읽기와 첫 탐색

학번처럼 **숫자처럼 보이지만 계산하지 않는 값**은 문자열로 읽어야 앞자리 0이 사라지지 않습니다. `dtype={"student_id": str}`

In [ ]:
df = pd.read_csv(get_data("students.csv"), dtype={"student_id": str})
df.head()

In [ ]:
df.info()

`info()`에서 확인할 것

- 행 수(`RangeIndex: 80 entries`)와 열마다 **Non-Null 개수** → `assignment`가 77개이니 **결측 3개**
- 자료형: `assignment`는 결측 때문에 정수가 아니라 `float64`가 됩니다.

In [ ]:
df.describe().round(1)

In [ ]:
print(df.isna().sum())
print()
print(df["department"].value_counts())
print()
print(df["year"].value_counts().sort_index())

In [ ]:
df.sample(3, random_state=1)    # 무작위 3행 (random_state로 재현 가능)

---
## 3. Level 1 따라하기: 선택 · 필터 · 정렬 · 새 열

### 열 선택

In [ ]:
df["final"].head(3)              # 열 하나 → Series
df[["name", "midterm", "final"]].head(3)   # 열 여러 개 → DataFrame (대괄호 두 겹)

### 행 선택: `loc`(이름) vs `iloc`(위치)

| 표현 | 의미 |
|---|---|
| `df.iloc[0]` | 0번째 행 |
| `df.iloc[:3, :2]` | 앞 3행, 앞 2열 (위치) |
| `df.loc[5, "name"]` | 인덱스 5인 행의 name |
| `df.loc[df["year"] == 1, ["name", "final"]]` | 조건 행의 특정 열 |

In [ ]:
print(df.iloc[0]["name"], "/", df.loc[5, "name"])
df.loc[df["year"] == 1, ["name", "department", "final"]].head()

### 조건 필터

NumPy와 마찬가지로 **`&`, `|`, `~`와 괄호**를 씁니다.

In [ ]:
cond = (df["department"] == "컴퓨터공학과") & (df["final"] >= 85)
df[cond][["name", "year", "final"]]

In [ ]:
print(df[df["department"].isin(["간호학과", "경영학과"])].shape)
print(df[df["final"].between(70, 79)].shape)                 # 70 이상 79 이하
print(df[df["assignment"].isna()][["name", "assignment"]])
df.query("year >= 3 and attendance < 12")[["name", "year", "attendance"]]

### 정렬

In [ ]:
df.sort_values(["final", "student_id"], ascending=[False, True])[["student_id", "name", "final"]].head()

In [ ]:
df.nlargest(3, "midterm")[["name", "midterm"]]

### 새 열 만들기

열끼리 연산하면 **모든 행에 한 번에** 계산됩니다 (NumPy 벡터화와 같음). 결측은 `fillna`로 계산용 값을 정합니다.

In [ ]:
df["total"] = (df["midterm"] * 0.3 + df["final"] * 0.4 + df["assignment"].fillna(0) * 0.3).round(1)
df["grade"] = pd.cut(df["total"], bins=[-1, 60, 70, 80, 90, 101], right=False, labels=["F", "D", "C", "B", "A"])
df.loc[df["attendance"] < 12, "grade"] = "F"        # 출석 규정
df[["name", "total", "attendance", "grade"]].head()

> `pd.cut(..., right=False)`는 구간을 `[60, 70)`처럼 **왼쪽 포함, 오른쪽 제외**로 나눕니다. 그래서 정확히 90.0점은 A입니다.

---
## 4. Level 2 바꾸기

### ✏️ 직접 해보기

1. `cond`를 바꿔 **정보보안학과이면서 출석 15회 이상**인 학생을 골라 보세요.
2. 정렬 기준을 `total` 내림차순, 같으면 `name` 오름차순으로 바꿔 상위 5명을 보세요.
3. `df["improved"] = df["final"] - df["midterm"]` 열을 만들고, 가장 많이 오른 3명을 찾아보세요.

In [ ]:
cond2 = (df["department"] == "컴퓨터공학과") & (df["attendance"] >= 15)   # ← 바꿔 보세요
print(df[cond2].shape)

df.sort_values("total", ascending=False)[["name", "total"]].head()        # ← 바꿔 보세요

---
## 5. Level 3 적용하기: 성적 데이터 탐색 보고

탐색(EDA)은 **질문**에서 출발합니다. 다음 다섯 질문에 코드로 답해 봅시다.

| 질문 | 방법 |
|---|---|
| Q1. 데이터는 몇 명, 몇 개 열이며 결측은 어디에 있나? | `shape`, `isna().sum()` |
| Q2. 등급 분포는? 8주차 중간 실기 결과와 같은가? | `value_counts` |
| Q3. 학과별 평균 총점과 통과율은? | `groupby` (맛보기) |
| Q4. 출석과 총점은 관련이 있나? | `corr`, 출석 구간별 평균 |
| Q5. 과제 미제출 학생은 누구이며 성적에 얼마나 영향이 있나? | 필터, 결측 처리 비교 |

In [ ]:
# Q1
print(f"{df.shape[0]}명, {df.shape[1]}개 열")
missing = df.isna().sum()
print("결측:", missing[missing > 0].to_dict())

In [ ]:
# Q2: 8주차 중간 실기 모범답안의 결과는 A 8 / B 22 / C 23 / D 12 / F 15
dist = df["grade"].value_counts().reindex(["A", "B", "C", "D", "F"])
print(dist.to_dict())
assert dist.to_dict() == {"A": 8, "B": 22, "C": 23, "D": 12, "F": 15}
print("8주차 결과와 일치 ✅  (수십 줄 → 몇 줄)")

In [ ]:
# Q3: groupby 맛보기 (11주차에 자세히)
df["passed"] = df["grade"] != "F"
summary = df.groupby("department").agg(
    count=("name", "size"),
    avg_total=("total", "mean"),
    pass_rate=("passed", "mean"),
).round(3)
summary["pass_rate"] = (summary["pass_rate"] * 100).round(1)
summary["avg_total"] = summary["avg_total"].round(1)
summary

In [ ]:
# Q4: 출석과 총점
print("상관계수:", round(df["attendance"].corr(df["total"]), 3))
df.groupby(pd.cut(df["attendance"], [0, 11, 13, 15, 16]), observed=True)["total"].agg(["count", "mean"]).round(1)

상관계수는 −1~1 사이 값으로, 1에 가까울수록 함께 커지는 경향이 강합니다. 하지만 **상관이 인과를 뜻하지는 않습니다.** (출석이 성적을 올렸는지, 성실한 학생이 둘 다 높은지는 이 데이터만으로 알 수 없습니다.)

In [ ]:
# Q5: 과제 미제출
no_assign = df[df["assignment"].isna()]
print(no_assign[["name", "department", "midterm", "final", "total", "grade"]])

# 미제출을 0점으로 볼 때 vs 학과 평균으로 채울 때 총점 차이
alt = (no_assign["midterm"] * 0.3 + no_assign["final"] * 0.4 + df["assignment"].mean() * 0.3).round(1)
print("평균으로 채운다면:", alt.tolist())

> **탐색 보고 정리 예**: 80명·8개 열, 과제 결측 3명. 등급 분포는 8주차 결과와 같다. 이 데이터에서 출석과 총점의 상관은 매우 약하다(약 0.05). 과제 미제출 3명은 0점 처리하면 과제 평균으로 채울 때보다 총점이 약 24점 낮아져, **결측을 어떻게 처리하느냐가 결론을 바꿀 수 있다**.

---
## 6. Level 4 AI와 함께: pandas 함정 두 가지

### 함정 1: 조건에 `and` 사용

In [ ]:
try:
    df[df["final"] > 80 and df["year"] >= 3]      # AI가 자주 쓰는 코드
except ValueError as e:
    print("ValueError:", str(e)[:70], "...")

ok = df[(df["final"] > 80) & (df["year"] >= 3)]
print("올바른 결과:", len(ok), "명")

### 함정 2: 체인 인덱싱으로 값 바꾸기

"1학년에게 보너스 5점을 주는 열을 만들어 줘"라는 요청에 AI가 준 코드입니다.

In [ ]:
import warnings

df["bonus"] = 0
with warnings.catch_warnings():
    warnings.simplefilter("ignore")              # 경고 메시지는 잠시 숨기고 결과만 확인
    df[df["year"] == 1]["bonus"] = 5             # AI 코드: 체인 인덱싱

print("1학년 보너스 합계:", df.loc[df["year"] == 1, "bonus"].sum())   # 0 → 바뀌지 않았다!

`df[조건]`이 **복사본**을 만들고, 그 복사본의 `bonus`를 바꾼 뒤 버려집니다. 원본은 그대로입니다. (pandas 버전에 따라 `SettingWithCopyWarning` 또는 `ChainedAssignmentError` 경고가 나옵니다.)

**해결**: 조건과 열을 `loc` **한 번에** 지정합니다.

In [ ]:
df.loc[df["year"] == 1, "bonus"] = 5
n_first = (df["year"] == 1).sum()
assert df.loc[df["year"] == 1, "bonus"].sum() == 5 * n_first
print(f"1학년 {n_first}명에게 보너스 적용 ✅")

### 📝 AI의 pandas 코드를 받을 때 확인할 것

| 확인 | 방법 |
|---|---|
| 조건에 `and`/`or`를 썼나? | `&`, `\|`와 괄호로 |
| `df[...][...] = 값` 형태가 있나? | `df.loc[행조건, 열] = 값` |
| 결과 행 수가 예상과 맞나? | `len()`, `shape`로 바로 확인 |
| 결측이 계산에서 어떻게 처리됐나? | `isna().sum()`, `fillna` 여부 |
| 원본을 바꾸는가, 새 DataFrame을 돌려주는가? | 함수라면 `df.copy()`로 시작 |

---
## 7. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 구조 | `Series`(값+인덱스), `DataFrame`(열들의 묶음) |
| 읽기 | `read_csv(..., dtype={"id": str})` |
| 첫 탐색 | `head`, `info`, `describe`, `isna().sum()`, `value_counts` |
| 선택 | `df["열"]`, `df[["열1", "열2"]]`, `loc`(이름), `iloc`(위치) |
| 필터 | `&` `\|` `~` 괄호, `isin`, `between`, `query` |
| 정렬 | `sort_values([...], ascending=[...])`, `nlargest` |
| 새 열 | 열 연산, `fillna`, `pd.cut`, `loc`으로 조건부 수정 |
| AI 검증 | `and` → `&`, 체인 인덱싱 → `loc`, 행 수·결측 확인 |

### 자기 점검

- [ ] `info()`에서 결측을 찾는 방법을 안다.
- [ ] `loc`과 `iloc`의 차이를 설명할 수 있다.
- [ ] `df[df.a > 1 and df.b < 2]`가 오류인 이유를 안다.
- [ ] `df[cond]["col"] = 값`이 원본을 바꾸지 못하는 이유와 해결책을 안다.

### 다음 주 예고

**Week 11 · 데이터 정제 · 변환 · 집계**: 표기가 제각각인 설문 데이터를 정제하고 `groupby`·`pivot_table`·`merge`로 집계합니다.